# Bird's Eye View (BEV) Visualization

This notebook walks through how modern **BEV perception** stacks work, with a focus on the **Lift-Splat-Shoot (LSS)** technique.

**What you will see:**
1. Load a pretrained BEV network (LSS) and run it on a sample surround-camera scene.
2. Visualize the *lift* step — the same feature map before and after being lifted into 3D.
3. Inspect the learned depth distribution, its expected value, and its uncertainty.
4. (Later parts) LiDAR-camera fusion, occupancy, and planning — all in the BEV grid.

> Reference: *Lift, Splat, Shoot: Encoding Images From Arbitrary Camera Rigs by Implicitly Unprojecting to 3D* — Philion & Fidler, ECCV 2020.

## Part 1 — Setup & load one real nuScenes sample

Four downloads: the LSS repo, the pretrained vehicle-segmentation weights, nuScenes mini, and the nuScenes map expansion.


In [ ]:
!pip install -q pyquaternion nuscenes-devkit efficientnet_pytorch==0.7.1 gdown
!git clone -q https://github.com/nv-tlabs/lift-splat-shoot.git
!echo "" > lift-splat-shoot/src/__init__.py
!gdown -q https://drive.google.com/uc?id=18fy-6beTFTZx5SrYLs9Xk7cY-fGSm7kw
!mkdir -p nuscenes-mini && wget -q https://www.nuscenes.org/data/v1.0-mini.tgz -O- | tar -xz -C nuscenes-mini
!wget -qO mapexp.zip https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/nuScenes-map-expansion-v1.3.zip
!unzip -qo mapexp.zip -d nuscenes-mini/maps
import sys; sys.path.insert(0, 'lift-splat-shoot')


In [ ]:
import torch
import numpy as np
from nuscenes.nuscenes import NuScenes
from src.models import LiftSplatShoot
from src.data   import SegmentationData
from src.tools  import denormalize_img

grid_conf = {'xbound': [-50., 50., 0.5], 'ybound': [-50., 50., 0.5],
             'zbound': [-10., 10., 20.], 'dbound': [4., 45., 1.]}
data_aug_conf = {
    'resize_lim': (0.193, 0.225), 'final_dim': (128, 352),
    'rot_lim':    (-5.4, 5.4),    'H': 900, 'W': 1600,
    'rand_flip':  True,           'bot_pct_lim': (0., 0.22),
    'cams': ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT'],
    'Ncams': 6,
}
CAMS     = data_aug_conf['cams']
imH, imW = data_aug_conf['final_dim']
device   = 'cuda' if torch.cuda.is_available() else 'cpu'

model = LiftSplatShoot(grid_conf, data_aug_conf, outC=1).to(device).eval()
model.load_state_dict(torch.load('model525000.pt', map_location='cpu'))

nusc = NuScenes('v1.0-mini', dataroot='nuscenes-mini', verbose=False)
ds   = SegmentationData(nusc, is_train=False, data_aug_conf=data_aug_conf, grid_conf=grid_conf)

SAMPLE_IDX = 30
imgs, rots, trans, intrins, post_rots, post_trans, gt = ds[SAMPLE_IDX]

pil_images = [denormalize_img(imgs[i]) for i in range(6)]
batch      = [x[None].to(device) for x in (imgs, rots, trans, intrins, post_rots, post_trans)]

print('loaded sample', SAMPLE_IDX, 'from scene',
      nusc.get('scene', ds.ixes[SAMPLE_IDX]['scene_token'])['name'])


In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl
import matplotlib.patches as mpatches
from PIL import Image
from src.tools import plot_nusc_map, add_ego
from nuscenes.map_expansion.map_api import NuScenesMap

with torch.no_grad():
    bev = model(*batch).sigmoid()[0, 0].cpu().numpy()

# 4 nuScenes cities + a name-to-city lookup for each scene
LOCATIONS = ['singapore-hollandvillage', 'singapore-queenstown',
             'boston-seaport',           'singapore-onenorth']
nusc_maps = {loc: NuScenesMap(dataroot='nuscenes-mini', map_name=loc) for loc in LOCATIONS}
scene2map = {s['name']: nusc.get('log', s['log_token'])['location'] for s in nusc.scene}

# BEV cell size (dx) and the ego-frame coord of cell (0, 0) (bx)
dx = np.array([grid_conf['xbound'][2], grid_conf['ybound'][2]])
bx = np.array([grid_conf['xbound'][0] + dx[0]/2, grid_conf['ybound'][0] + dx[1]/2])

val = 0.01
fig = plt.figure(figsize=(3 * imW * val, (1.5 * imW + 2 * imH) * val))
gs  = mpl.gridspec.GridSpec(3, 3, height_ratios=(1.5 * imW, imH, imH))
gs.update(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)

for i, cam in enumerate(CAMS):
    ax  = plt.subplot(gs[1 + i // 3, i % 3])
    img = pil_images[i].transpose(Image.FLIP_LEFT_RIGHT) if i > 2 else pil_images[i]
    plt.imshow(img); plt.axis('off')
    plt.annotate(cam.replace('_', ' '), (0.01, 0.92), xycoords='axes fraction', color='white')

ax = plt.subplot(gs[0, :])
plt.imshow(bev, vmin=0, vmax=1, cmap='Blues')
plot_nusc_map(ds.ixes[SAMPLE_IDX], nusc_maps, nusc, scene2map, dx, bx)
add_ego(bx, dx)
plt.xlim(bev.shape[1], 0); plt.ylim(0, bev.shape[0])
plt.xticks([]); plt.yticks([])
plt.setp(ax.spines.values(), color='b', linewidth=2)
plt.legend(handles=[
    mpatches.Patch(color='blue',                label='Predicted vehicles'),
    mpatches.Patch(color='#76b900',             label='Ego'),
    mpatches.Patch(color=(1., 0.5, 0.31, 0.8),  label='Map'),
])
plt.show()


## Part 2 — Visualizing the **Lift** step

> **Heads up on the input data.** The previous cells use a synthetic surround-camera scene so the notebook can run anywhere. NVIDIA's pretrained LSS weights *are* loaded, but the depth-net has never seen toy images like this, so the predicted depths/BEV mask are not meaningful. The *mechanics* of the lift, however, are identical — which is exactly what we want to visualize here. To swap in a real nuScenes sample, install `nuscenes-devkit`, download the `v1.0-mini` split, and replace the `pil_images` / `rots` / `trans` / `intrins` tensors with the ones returned by `NuscData.__getitem__` (see `lift-splat-shoot/src/data.py`).

The lift turns every pixel feature into a *ray of features* — one per depth bin — weighted by a learned categorical distribution over depth. Concretely:

```
for each pixel (u, v):
    f(u, v)      ∈ R^C            # context feature vector
    α(u, v, d)   ∈ Δ^D            # softmax over D depth bins
    lifted(u, v, d) = α(u, v, d) · f(u, v)    ∈ R^C
```

We now extract these intermediate tensors from the pretrained `CamEncode` module.

In [ ]:
# Extract intermediate tensors from the CamEncode module for all 6 cameras
imgs_b, rots_b, trans_b, intrins_b, post_rots_b, post_trans_b = batch
B, N, _, imH, imW = imgs_b.shape
imgs_flat = imgs_b.view(B * N, 3, imH, imW)

with torch.no_grad():
    eff_feat     = model.camencode.get_eff_depth(imgs_flat)   # (N, 512, fH, fW)
    depth_logits = model.camencode.depthnet(eff_feat)         # (N, D+C, fH, fW)
    D_ = model.camencode.D
    C_ = model.camencode.C
    depth_dist = depth_logits[:, :D_].softmax(dim=1)          # (N, D, fH, fW)
    ctx_feat   = depth_logits[:, D_:]                         # (N, C, fH, fW)
    lifted     = depth_dist.unsqueeze(1) * ctx_feat.unsqueeze(2)  # (N, C, D, fH, fW)

fH, fW = eff_feat.shape[-2:]
print(f'Feature grid per camera : {fH} x {fW}   (input {imH}x{imW}, stride {imH//fH})')
print(f'Depth bins              : D = {D_}  from {grid_conf["dbound"][0]} m to {grid_conf["dbound"][1]} m')
print(f'Context feature channels: C = {C_}')
print('')
print(f'2D features before lift : ctx_feat    {tuple(ctx_feat.shape)}')
print(f'Depth distribution      : depth_dist  {tuple(depth_dist.shape)}')
print(f'Lifted 3D frustum feats : lifted      {tuple(lifted.shape)}   # = depth_dist ⊗ ctx_feat')

depth_bins = torch.arange(*grid_conf["dbound"], dtype=torch.float32, device=device)


### 2.1 — Features **before** the lift  (2D)

The `CamEncode` module turns each camera image into a dense `(C=64, fH=8, fW=22)` feature map. We PCA those 64 channels down to 3 RGB channels so the pattern is visible. Same colour ≈ the CNN thinks those regions are semantically similar.

**Keep this colour pattern in your head.** In section 2.4 you will see the *same* colours again, but spread in 3D along each pixel's depth ray — that's exactly what the lift does.


In [ ]:
from sklearn.decomposition import PCA
from PIL import Image as PILImage

def feat_to_rgb(feat_2d):
    """(C, H, W) tensor -> (H, W, 3) RGB in [0, 1] via per-pixel PCA."""
    C_, H_, W_ = feat_2d.shape
    X = feat_2d.reshape(C_, -1).T.cpu().numpy()
    Y = PCA(n_components=3).fit_transform(X)
    Y = (Y - Y.min(0)) / (Y.max(0) - Y.min(0) + 1e-6)
    return Y.reshape(H_, W_, 3)

def upscale_rgb(rgb, out_H, out_W):
    """Bicubic-upscale an (H, W, 3) RGB array to (out_H, out_W, 3)."""
    img = PILImage.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8))
    return np.asarray(img.resize((out_W, out_H), PILImage.BICUBIC)) / 255.0

# Compute PCA features for every camera and upscale to the full image size so that
# spatial correspondence with the input is visible (the raw feature maps are only 8×22).
feat_rgbs = [upscale_rgb(feat_to_rgb(ctx_feat[i]), imH, imW) for i in range(len(CAMS))]

fig, axes = plt.subplots(2, 6, figsize=(17, 5.2))
for i, cam in enumerate(CAMS):
    axes[0, i].imshow(pil_images[i]);    axes[0, i].set_title(cam, fontsize=9); axes[0, i].axis('off')
    axes[1, i].imshow(feat_rgbs[i]);                                          axes[1, i].axis('off')

# Row labels on the far left
fig.text(0.012, 0.73, 'INPUT\nRGB image\n(what the\ncamera sees)',
         fontsize=10, ha='left', va='center',
         bbox=dict(boxstyle='round', facecolor='#d8ecff', edgecolor='#4a78b5'))
fig.text(0.012, 0.27, 'CNN\nFEATURES\n(64-d\u2192RGB\nvia PCA)',
         fontsize=10, ha='left', va='center',
         bbox=dict(boxstyle='round', facecolor='#efdfff', edgecolor='#7d4bb5'))

fig.suptitle("BEFORE the lift \u2014 what the 2D CNN 'sees'\n"
             "The EfficientNet backbone turns each 16\u202f\u00d7\u202f16 image region into a 64-d feature vector.\n"
             "PCA reduces those 64 numbers to 3 RGB channels, then we upscale back to image resolution.\n"
             "Same color = the CNN thinks those regions are semantically similar (road / car / building / sky / \u2026).",
             fontsize=10.5, y=1.09)
plt.tight_layout(rect=[0.06, 0, 1, 1])
plt.show()

print(f'Feature map shape per camera: {ctx_feat.shape[1]} channels \u00d7 {fH} \u00d7 {fW}   '
      f'(displayed upscaled to {imH} \u00d7 {imW})')

### 2.2 — Depth distribution per pixel — **interactive explorer**

For every cell of the downsampled feature map, LSS predicts a categorical distribution over `D = 41` depth bins (4 m … 45 m, step 1 m).

The cell below is **interactive**: slide the *row* / *col* knobs (or switch camera) to pick any feature cell — the depth PDF on the right redraws live.

**What to look for**

- *Sharp single peak*  →  the network is confident about depth.
- *Broad / bi-modal PDF*  →  the network is unsure (multiple plausible depths).
- Pixels on the **road surface** tend to peak near a specific depth.
- Pixels in the **sky** or on **very distant** objects typically show much broader distributions.


In [ ]:
# Interactive depth-distribution explorer
# ------------------------------------------------------------------------
# LSS predicts a full 41-bin probability distribution over depth for every
# 16x16 region of the image. Below, slide the row/column sliders (or change
# camera) to pick any one of those cells and see its distribution live.
# The red box on the left shows which 16x16 patch you're inspecting.

from matplotlib.patches import Rectangle

try:
    from ipywidgets import interact, IntSlider, Dropdown
except ImportError:
    !pip install -q ipywidgets
    from ipywidgets import interact, IntSlider, Dropdown

_depth_bins_np = depth_bins.cpu().numpy()
_stride_y, _stride_x = imH // fH, imW // fW

def _depth_explorer(cam='CAM_FRONT', fh=fH // 2, fw=fW // 2):
    c   = CAMS.index(cam)
    pdf = depth_dist[c, :, fh, fw].cpu().numpy()
    e_d = float((pdf * _depth_bins_np).sum())
    h_d = float(-(pdf * np.log(pdf + 1e-12)).sum())

    u_px = fw * _stride_x + _stride_x // 2
    v_px = fh * _stride_y + _stride_y // 2

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.3))
    ax0.imshow(pil_images[c])
    ax0.add_patch(Rectangle((fw * _stride_x, fh * _stride_y), _stride_x, _stride_y,
                            linewidth=2.5, edgecolor='red', facecolor='none'))
    ax0.scatter([u_px], [v_px], c='red', s=80, edgecolor='white', linewidths=2, zorder=5)
    ax0.set_title(f'{cam}  —  selected feature cell (row={fh}, col={fw})  ≈  image pixel ({v_px}, {u_px})',
                  fontsize=10)
    ax0.axis('off')

    ax1.bar(_depth_bins_np, pdf, width=0.9, color='steelblue', edgecolor='navy', alpha=0.8)
    ax1.axvline(e_d, color='limegreen', lw=2.2, label=f'E[d] = {e_d:.1f} m')
    ax1.set_xlabel('depth bin  [m]'); ax1.set_ylabel('P(depth)')
    ax1.set_title(f'Depth distribution at this cell   (entropy = {h_d:.2f} nats)', fontsize=10)
    ax1.set_xlim(3.5, 45.5)
    ax1.set_ylim(0, max(pdf.max() * 1.15, 0.05))
    ax1.legend(); ax1.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print(f'Feature grid per camera: {fH} rows × {fW} cols   (each cell ↔ a {_stride_y} × {_stride_x} image patch)')
interact(
    _depth_explorer,
    cam=Dropdown(options=CAMS, value='CAM_FRONT', description='camera'),
    fh=IntSlider(min=0, max=fH - 1, step=1, value=fH // 2, description='row (fh)'),
    fw=IntSlider(min=0, max=fW - 1, step=1, value=fW // 2, description='col (fw)'),
);


### 2.3 — A static snapshot: three pixels side-by-side

If you'd rather see a static picture (e.g. when the notebook is rendered on nbviewer where sliders don't work), here are the depth PDFs at three hand-picked pixels of the current CAM_FRONT frame.


In [ ]:
cam_idx = CAMS.index("CAM_FRONT")
dd      = depth_dist[cam_idx]   # (D, fH, fW)

sample_pixels = [
    (fH // 4,     fW // 2),   # upper-middle (far)
    (fH // 2,     fW // 3),   # mid-left
    (3 * fH // 4, 2 * fW // 3),  # lower-right (close)
]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4))
ax0.imshow(pil_images[cam_idx]); ax0.set_title(f'{CAMS[cam_idx]} — sampled pixels'); ax0.axis('off')
stride_y, stride_x = imH // fH, imW // fW
for (yy, xx) in sample_pixels:
    ax0.scatter([xx * stride_x + stride_x/2], [yy * stride_y + stride_y/2], s=120, edgecolor='white', linewidth=2)
    ax1.plot(depth_bins.cpu(), dd[:, yy, xx].cpu(), marker='o', label=f'pixel ({yy}, {xx})')
ax1.set_xlabel('depth [m]'); ax1.set_ylabel('P(depth)')
ax1.set_title('Categorical depth distribution at each marked pixel')
ax1.legend(); ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 2.4 — Features **after** the lift  (3D)

The lift takes the 2D feature map from 2.1 and makes a 3D frustum of shape `(C, D, fH, fW)` by multiplying every feature vector by the per-pixel depth distribution:

$$\text{lifted}(c, d, h, w) \;=\; f_c(h, w) \cdot \alpha_d(h, w)$$

Concretely: for each 2D pixel we get **D=41 copies** of its feature vector, one per depth bin, weighted by the probability the network assigns to that bin.

In the 3D scatter below:

- **Colour = the same PCA→RGB as in 2.1** — so a cell keeps its identity when it is reprojected to 3D.
- **Alpha = P(depth)** — bins the network thinks are likely look solid, bins it doesn't think are likely fade out.

Run the cell and compare with 2.1: you'll see the 2D colour grid stretched along depth rays.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # registers the 3d projection

cam_idx = CAMS.index('CAM_FRONT')

# Geometry: (x, y, z) ego-frame coord of every (d, h, w) voxel for this camera
with torch.no_grad():
    geom = model.get_geometry(*batch[1:])
geom_cam  = geom[0, cam_idx].cpu().numpy()            # (D, fH, fW, 3)
depth_cam = depth_dist[cam_idx].cpu().numpy()         # (D, fH, fW)

# SAME PCA colours as 2.1 (ctx_feat is deterministic in, feat_to_rgb is deterministic out)
rgb_2d = feat_to_rgb(ctx_feat[cam_idx])               # (fH, fW, 3)
D_, fH_, fW_ = depth_cam.shape

# Broadcast the 2D colours along the depth axis — every voxel (d, h, w) gets
# the colour of its source pixel (h, w). The depth probability becomes alpha.
rgb_3d = np.broadcast_to(rgb_2d[None], (D_, fH_, fW_, 3)).reshape(-1, 3)
alphas = depth_cam.reshape(-1)
points = geom_cam.reshape(-1, 3)

# Drop voxels the network thinks are very unlikely so the scatter stays legible.
keep   = alphas > 0.02
points = points[keep]; rgb_3d = rgb_3d[keep]; alphas = alphas[keep]

# Scale alpha into a visible range for matplotlib
alphas_plot = np.clip(alphas * 4, 0.05, 1.0)
rgba        = np.concatenate([rgb_3d, alphas_plot[:, None]], axis=1)

fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=rgba, s=8)
ax.scatter([0], [0], [0], c='cyan', marker='^', s=150, edgecolor='black', label='ego')
ax.set_xlabel('x (forward) [m]')
ax.set_ylabel('y (left) [m]')
ax.set_zlabel('z (up) [m]')
ax.set_title(f'{CAMS[cam_idx]} frustum AFTER the lift\n'
             f'same PCA colours as 2.1,  alpha = P(depth)')
ax.view_init(elev=22, azim=-65)
ax.legend(); plt.tight_layout(); plt.show()

print(f'Voxels drawn: {len(points):,}  (out of {D_ * fH_ * fW_:,} total in this camera\'s frustum)')


### 2.5 — The **splat** — pooling all 6 cameras into one BEV grid

The final step is *splat*: every lifted voxel is scattered into its corresponding cell of a 2D BEV grid. Voxels that fall into the same cell are summed (the "cumulative-sum trick" in the LSS paper). This gives a single feature tensor `(C, 200, 200)` covering a 100 m × 100 m region around the ego vehicle.

We show two views of that tensor:
1. the L2 magnitude of the per-cell feature vector (where does LSS place *any* feature),
2. a PCA → RGB projection (what kind of features were placed where).

In [ ]:
with torch.no_grad():
    x_cam_feats = model.get_cam_feats(imgs_b)                    # (B, N, D, fH, fW, C)
    bev_feat    = model.voxel_pooling(geom, x_cam_feats)          # (B, C, X, Y)

bev_mag = bev_feat[0].norm(dim=0).cpu().numpy()                  # (X, Y)
bev_rgb = feat_to_rgb(bev_feat[0].cpu())                          # (X, Y, 3)

# to_topdown: transpose + flipud so that ego-X (forward) runs horizontally
# (right = forward) and ego-Y (left) runs vertically (up = left).
def to_topdown(arr):
    return np.flipud(arr.transpose(1, 0) if arr.ndim == 2 else arr.transpose(1, 0, 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.8))
extent = [-50, 50, -50, 50]
axes[0].imshow(to_topdown(bev_mag), cmap='magma', extent=extent)
axes[0].set_title('BEV feature magnitude ||f||₂')
axes[1].imshow(to_topdown(bev_rgb),                 extent=extent)
axes[1].set_title('BEV features  (PCA → RGB)')
for ax in axes:
    ax.scatter([0], [0], c='cyan', marker='^', s=180, edgecolor='black', label='ego')
    # Forward arrow: with this transposition, 'forward' points to the RIGHT of the plot.
    ax.annotate('', xy=(10, 0), xytext=(1.5, 0),
                arrowprops=dict(arrowstyle='-|>', color='cyan', lw=2.2, mutation_scale=18))
    ax.text(11.5, 0, 'forward', color='cyan', fontsize=9, va='center')
    ax.set_xlabel('x  [m]   (+ = forward)')      # horizontal = ego-X
    ax.set_ylabel('y  [m]   (+ = left of ego)')  # vertical   = ego-Y
    ax.set_xlim(-18, 18); ax.set_ylim(-18, 18)   # zoom in — feats are only ~15 m around the ego
    ax.set_aspect('equal')
    ax.legend(loc='upper right')
plt.suptitle('Splatted BEV features from all 6 cameras   (zoomed to ±18 m around the ego)', y=1.02)
plt.tight_layout(); plt.show()


**Recap of Part 2.** We walked through every stage of a Lift-Splat-Shoot forward pass:

| Stage | Tensor shape | What it means |
|------|--------------|---------------|
| 2D image features | `(N, C, fH, fW)` | per-pixel context descriptor |
| Depth distribution | `(N, D, fH, fW)` | `p(depth)` per pixel |
| Lifted 3D frustum | `(N, C, D, fH, fW)` | outer product `α ⊗ f` |
| Voxel-pooled BEV | `(1, C, 200, 200)` | everything summed into one BEV grid |

Up next — **Part 3**: using these same building blocks to look at depth *uncertainty* across the scene and compare it against ground truth when available. Then Parts 4–6 cover LiDAR-camera fusion, occupancy, and planning, all in the same BEV grid.

## Part 3 — **Shoot**: planning on a BEV cost map

**What the paper actually does (Section 3.3 of Philion & Fidler, 2020):**

1. Run k-means with **K = 1000** on ego-motion trajectories from the nuScenes training set to get a bank of template trajectories $\mathcal{T}$.
2. Add a **separate planning head** on top of the BEV features that outputs a cost map $c_o(x, y)$ per cell.
3. Train that head end-to-end with **cross-entropy** against the template closest to each ground-truth trajectory, using the softmax

   $$p(\tau \mid o) = \frac{\exp\big(-\sum_{(x,y)\in\tau} c_o(x,y)\big)}{\sum_{\tau' \in \mathcal{T}} \exp\big(-\sum_{(x,y)\in\tau'} c_o(x,y)\big)}.$$

4. At inference, pick the template with the **lowest** path-integrated cost.

**What we can actually run here.** Neither the planning head's weights nor the 1000 k-means templates are released — the only pretrained file (`model525000.pt`) is the **vehicle-segmentation** head (see `src/data.py :: get_binimg`, which rasterises vehicle silhouettes). So this section is an **explicit approximation**:

- We **substitute** the planning cost map with the released vehicle-segmentation BEV. A cell with high $P(\text{vehicle})$ becomes expensive, a clear cell becomes cheap:  $c(x, y) = -\log\big(1 - P(\text{vehicle})\big).$
- We **substitute** the 1000 k-means templates with a small hand-crafted bank of 17 smooth lane-change candidates.

The *mechanism* (sum the cell costs along each template, take the argmin) is identical to the paper; the cost map and template bank are not.

> If you want to re-do this properly for a research project: retrain LSS with a planning head and cross-entropy on expert trajectories, or load a more recent planner (e.g. UniAD / VAD) instead of the LSS vehicle-segmentation weights.


### 3.1 — A bank of candidate trajectories

We sample **17 smooth paths** that all go 30 m forward but end at different lateral offsets, from −8 m (ego's right) to +8 m (ego's left). Each path starts at the ego with zero initial heading and uses a cubic-hermite blend (`3t² − 2t³`) for lateral displacement — so the ego never has to swerve instantly.

In [ ]:
from matplotlib.gridspec import GridSpec

# --- BEV display helper (forward up, ego-left on the left) ------------------
BEV_EXTENT = [-50, 50, -50, 50]
def bev_show(ax, arr, **kw):
    kw.setdefault('extent', BEV_EXTENT)
    im = ax.imshow(arr, origin='lower', **kw)
    ax.invert_xaxis()
    ax.set_xlabel('y  [m]   (+ = left of ego)')
    ax.set_ylabel('x  [m]   (+ = forward)')
    ax.set_aspect('equal')
    return im

def smooth_traj(x_end, y_end, length_pts=40):
    """Cubic-hermite-smoothed path from (0, 0) with zero initial heading to (x_end, y_end)."""
    t = np.linspace(0, 1, length_pts)
    x = x_end * t
    y = y_end * (3 * t**2 - 2 * t**3)
    return np.stack([x, y], axis=1)

# 17 hand-crafted lane-change candidates, ±8 m lateral at 30 m forward.
# (Stand-in for the paper's 1000 k-means templates, not released with these weights.)
LATERAL_TARGETS = np.linspace(-8.0, 8.0, 17)
FORWARD_TARGET  = 30.0
trajectories    = [smooth_traj(FORWARD_TARGET, lat) for lat in LATERAL_TARGETS]

fig, ax = plt.subplots(figsize=(7.5, 7.5))
bev_show(ax, bev, cmap='Blues', vmin=0, vmax=1, alpha=0.9)
for tr in trajectories:
    ax.plot(tr[:, 1], tr[:, 0], color='gray', alpha=0.55, lw=1.3)
ax.scatter([0], [0], c='cyan', marker='^', s=220, edgecolor='black', zorder=5, label='ego')
ax.set_title(f'3.1 — {len(trajectories)} candidate trajectories ({FORWARD_TARGET:.0f} m ahead, ±{LATERAL_TARGETS.max():.0f} m lateral)',
             fontsize=11)
ax.set_xlim(18, -18); ax.set_ylim(-5, 35)
ax.legend(loc='lower left'); plt.tight_layout(); plt.show()


### 3.2 — Score each candidate on the vehicle BEV (approximated cost map)

For every candidate τ we look up the per-cell $P(\text{vehicle})$ along the path and sum the per-cell cost

$$c(\tau) = -\frac{1}{|\tau|} \sum_{(x, y)\in\tau} \log\big(1 - P(\text{vehicle at }(x, y))\big).$$

- A completely clear path  →  $P(\text{vehicle}) \approx 0$  →  $c(\tau) \approx 0$.
- A path that runs through a car  →  $P(\text{vehicle}) \to 1$  →  $c(\tau) \to +\infty$.

This is the same functional form the paper uses in its softmax over trajectories; only the source of the cost map differs (paper: learned planning head; us: released vehicle-segmentation head).


In [ ]:
# --- paper-accurate cost ----------------------------------------------------
# LSS §5.1 uses  c(τ) = -Σ log P(no-vehicle at cell)  over the trajectory cells.
# The pretrained head outputs P(VEHICLE), so we need  1 - P(vehicle)  inside the log.
# The template bank in the paper is 1000 trajectories mined by k-means from the
# nuScenes training set — not released with the pretrained weights — so we
# substitute with our small hand-crafted bank from 3.1.

def trajectory_cost(traj, bev, grid_conf):
    xmin, _, xres = grid_conf['xbound']
    ymin, _, yres = grid_conf['ybound']
    xi = np.floor((traj[:, 0] - xmin) / xres).astype(int)
    yi = np.floor((traj[:, 1] - ymin) / yres).astype(int)
    if ((xi < 0) | (xi >= bev.shape[0]) | (yi < 0) | (yi >= bev.shape[1])).any():
        return float('inf')
    p = bev[xi, yi]                                    # P(vehicle) per cell
    return float(-np.mean(np.log(1.0 - p + 1e-3)))     # penalise cells likely occupied

costs    = np.array([trajectory_cost(tr, bev, grid_conf) for tr in trajectories])
best_idx = int(np.argmin(costs))
print(f'Best candidate: lateral target = {LATERAL_TARGETS[best_idx]:+.1f} m   NLL = {costs[best_idx]:.3f}')

fig, ax = plt.subplots(figsize=(9, 3.3))
bars = ax.bar(LATERAL_TARGETS, costs, width=1.3, color='steelblue', edgecolor='black')
bars[best_idx].set_color('limegreen')
ax.set_xlabel('lateral target at 30 m  [m]   (+ = left of ego)')
ax.set_ylabel('cost  =  -mean log P(no vehicle)')
ax.set_title(f'3.2 — Cost per candidate   (green bar = lowest-cost winner)')
ax.invert_xaxis()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


### 3.3 — The winning trajectory

We color-code all candidates by their cost (green = low, red = high), highlight the winner in lime, and — as a sanity check — project it back onto the **CAM_FRONT** image so you can visually verify it lands on actual road.

In [ ]:
# Project a trajectory (z = 0) into the CAM_FRONT image plane using
# the same intrinsics / extrinsics / post-aug transforms that LSS used.
def project_to_cam(ego_xy, cam_idx, batch, imH, imW):
    R  = batch[1][0, cam_idx].cpu().numpy()       # cam_to_ego rotation
    t  = batch[2][0, cam_idx].cpu().numpy()       # cam_to_ego translation
    K  = batch[3][0, cam_idx].cpu().numpy()       # camera intrinsics
    pr = batch[4][0, cam_idx].cpu().numpy()       # post-aug rotation (3x3)
    pt = batch[5][0, cam_idx].cpu().numpy()       # post-aug translation (3,)

    pts_ego = np.column_stack([ego_xy[:, 0], ego_xy[:, 1], np.zeros(len(ego_xy))])
    pts_cam = (pts_ego - t) @ R                   # ego -> cam  (R orthonormal, so R^-1 = R^T)
    keep = pts_cam[:, 2] > 0.1
    pts_cam = pts_cam[keep]
    if len(pts_cam) < 2:
        return np.array([]), np.array([])
    uv = pts_cam @ K.T
    u0, v0 = uv[:, 0] / uv[:, 2], uv[:, 1] / uv[:, 2]
    u = pr[0, 0] * u0 + pr[0, 1] * v0 + pt[0]
    v = pr[1, 0] * u0 + pr[1, 1] * v0 + pt[1]
    ok = (u >= 0) & (u < imW) & (v >= 0) & (v < imH)
    return u[ok], v[ok]

# --- side-by-side: BEV view (left) + CAM_FRONT projection (right) --------
norm_costs = (costs - costs.min()) / (costs.max() - costs.min() + 1e-8)
cmap_rgb   = plt.get_cmap('RdYlGn_r')

fig = plt.figure(figsize=(15, 6.8))
gs  = GridSpec(1, 2, width_ratios=[1, 1.35])

ax_bev = fig.add_subplot(gs[0])
bev_show(ax_bev, bev, cmap='Blues', alpha=0.85, vmin=0, vmax=1)
for i, tr in enumerate(trajectories):
    if i == best_idx: continue
    ax_bev.plot(tr[:, 1], tr[:, 0], color=cmap_rgb(norm_costs[i]), alpha=0.55, lw=1.8)
winner = trajectories[best_idx]
ax_bev.plot(winner[:, 1], winner[:, 0], color='lime',      lw=4.5, zorder=6, label=f'winner  ({LATERAL_TARGETS[best_idx]:+.1f} m)')
ax_bev.plot(winner[:, 1], winner[:, 0], color='darkgreen', lw=2.0, ls='--', zorder=7)
ax_bev.scatter([winner[-1, 1]], [winner[-1, 0]], c='lime', marker='*', s=350,
               edgecolor='black', zorder=8, label='goal')
ax_bev.scatter([0], [0], c='cyan', marker='^', s=220, edgecolor='black', zorder=5, label='ego')
ax_bev.set_title('BEV view — candidates colored by cost\n(green = low cost, red = high cost)',
                 fontsize=11)
ax_bev.set_xlim(18, -18); ax_bev.set_ylim(-5, 35)
ax_bev.legend(loc='lower left', fontsize=9)

ax_cam = fig.add_subplot(gs[1])
ax_cam.imshow(pil_images[CAMS.index('CAM_FRONT')])
u, v = project_to_cam(winner, CAMS.index('CAM_FRONT'), batch, imH, imW)
if len(u) >= 2:
    ax_cam.plot(u, v, '-', color='darkgreen', lw=5)
    ax_cam.plot(u, v, '-', color='lime',      lw=2.5)
    ax_cam.scatter(u[::4], v[::4], c='lime', s=70, edgecolor='black', zorder=5)
ax_cam.set_title('3.3 — Winning trajectory projected onto CAM_FRONT\n(sanity check: does it land on road?)',
                 fontsize=11)
ax_cam.axis('off')

fig.suptitle('Lift → Splat → **Shoot**  —  the pretrained LSS weights are enough to run the full pipeline on a single keyframe.',
             fontsize=12, y=1.03)
plt.tight_layout(); plt.show()


## Part 4 — Full-scene replay (video)

nuScenes mini has 2 Hz keyframes, ~40 per scene. We run LSS on every keyframe of one scene and animate the output — CAM_FRONT on the left, predicted-vehicle BEV + map on the right.

In [ ]:
from tqdm import tqdm

# Pick the first scene in mini_val; list its keyframe indices into ds.ixes
scene       = nusc.scene[0]
scene_idx   = [i for i, r in enumerate(ds.ixes) if r['scene_token'] == scene['token']]
print(f'{len(scene_idx)} keyframes in scene {scene["name"]!r}')

# Run LSS on every keyframe; cache BEV + denormed CAM_FRONT for the animation
bevs, fronts = [], []
front_i = CAMS.index('CAM_FRONT')
for i in tqdm(scene_idx):
    imgs, rots, trans, intrins, post_rots, post_trans, _ = ds[i]
    b = [x[None].to(device) for x in (imgs, rots, trans, intrins, post_rots, post_trans)]
    with torch.no_grad():
        bevs.append(model(*b).sigmoid()[0, 0].cpu().numpy())
    fronts.append(denormalize_img(imgs[front_i]))


In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, (ax_cam, ax_bev) = plt.subplots(1, 2, figsize=(11, 5))
im_cam = ax_cam.imshow(fronts[0]); ax_cam.axis('off'); ax_cam.set_title('CAM_FRONT')
ax_bev.set_xticks([]); ax_bev.set_yticks([]); ax_bev.set_title('LSS vehicle BEV + map')

def update(i):
    im_cam.set_data(fronts[i])
    ax_bev.clear()
    ax_bev.imshow(bevs[i], vmin=0, vmax=1, cmap='Blues')
    plot_nusc_map(ds.ixes[scene_idx[i]], nusc_maps, nusc, scene2map, dx, bx)
    add_ego(bx, dx)
    ax_bev.set_xlim(bevs[i].shape[1], 0); ax_bev.set_ylim(0, bevs[i].shape[0])
    ax_bev.set_xticks([]); ax_bev.set_yticks([])
    ax_bev.set_title(f'LSS vehicle BEV  —  frame {i+1}/{len(bevs)}')
    return []

anim = animation.FuncAnimation(fig, update, frames=len(bevs), interval=500, blit=False)
plt.close()
HTML(anim.to_jshtml())


### Wrap-up

| Stage | What you saw | Source of truth |
|------|-------------|-----------------|
| **Lift**  | 2D features per camera, categorical depth PDF, outer-product frustum | `CamEncode` in `src/models.py` |
| **Splat** | 6 frustums → one `(C=64, 200, 200)` BEV tensor via cumsum pooling | `LiftSplatShoot.voxel_pooling` |
| **Shoot** (approximated) | 17 candidate trajectories scored on the vehicle BEV; lowest-cost winner | paper §3.3 in spirit only — the real Shoot uses a separate planning head + 1000 k-means templates, neither released |

**Honest caveats for the classroom:**

- The pretrained weights (`model525000.pt`) are the vehicle-segmentation head. No drivable-area, no planning, no LiDAR fusion.
- Every Shoot result here is therefore a proxy: the cost map only says "where other vehicles are", not "where the expert would drive". For a correct `p(τ|o)` you'd retrain LSS with a planning head — or switch to a planner checkpoint (UniAD, VAD, GenAD, …) in a follow-up lab.

**Follow-up labs in this DLC:**

- **Lab 2 — BEV Fusion / BEVFormer.** Add LiDAR (proper early fusion) or learned BEV queries with deformable cross-attention.
- **Lab 3 — SurroundOcc / VoxFormer.** Full 3-D occupancy prediction with height slices.
